# 🛡️ News Sentinel — End-to-End NLP Notebook
**Sentiment Analysis & Fake News Detection**

This notebook walks through the complete pipeline:
1. Data loading & generation
2. NLP preprocessing
3. Feature engineering
4. Model training & evaluation
5. Visualizations
6. Inference demo

In [ ]:
# Setup — run from project root
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Pretty plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Setup complete ✓')

## 1. Data Loading

In [ ]:
from src.data_preprocessing import load_dataset, generate_sample_dataset

# Load or generate data
df = load_dataset(data_dir='../data/')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#00cc66', '#ff4444'])
axes[0].set_title('Fake vs Real Distribution')
axes[0].set_xlabel('Label')
axes[0].tick_params(axis='x', rotation=0)

if 'sentiment' in df.columns:
    df['sentiment'].value_counts().plot(kind='pie', ax=axes[1],
        colors=['#00cc66', '#4a9eff', '#ff4444'], autopct='%1.1f%%')
    axes[1].set_title('Sentiment Distribution')
    axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 2. NLP Preprocessing

In [ ]:
from src.data_preprocessing import TextPreprocessor, assign_sentiment_labels

# Initialize preprocessor
preprocessor = TextPreprocessor()

# Demo on a sample text
sample = "BREAKING: Scientists at NASA have discovered SHOCKING evidence of extraterrestrial life!"
print('Original:', sample)
print()
print('Step 1 - Lowercase:', preprocessor.lowercase(sample))
print('Step 2 - Remove URLs:', preprocessor.remove_urls(sample.lower()))
print('Step 3 - Remove punct:', preprocessor.remove_punctuation(sample.lower()))
tokens = preprocessor.tokenize(preprocessor.remove_punctuation(sample.lower()))
print('Step 4 - Tokenized:', tokens[:10])
filtered = preprocessor.remove_stopwords(tokens)
print('Step 5 - No stopwords:', filtered)
lemmatized = preprocessor.lemmatize(filtered)
print('Step 6 - Lemmatized:', lemmatized)
print()
print('Final clean text:', preprocessor.preprocess(sample))

In [ ]:
# Run full preprocessing
from src.data_preprocessing import run_preprocessing_pipeline

df_processed = run_preprocessing_pipeline(data_dir='../data/')
print(f'Processed dataset: {df_processed.shape}')
df_processed[['label', 'sentiment', 'clean_text']].head(3)

## 3. Feature Engineering

In [ ]:
from src.feature_engineering import TFIDFFeatureBuilder, get_word_frequencies
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df_processed['clean_text'], df_processed['label'],
    test_size=0.2, random_state=42, stratify=df_processed['label']
)

# Build TF-IDF features
tfidf_builder = TFIDFFeatureBuilder(max_features=30_000, ngram_range=(1, 2))
X_train_tfidf = tfidf_builder.fit_transform(X_train)
X_test_tfidf = tfidf_builder.transform(X_test)

print(f'Training matrix: {X_train_tfidf.shape}')
print(f'Test matrix:     {X_test_tfidf.shape}')
print(f'Top 10 features: {tfidf_builder.get_top_features(10)}')

In [ ]:
# Word frequency analysis
freq_df = get_word_frequencies(df_processed['clean_text'], top_n=20)

plt.figure(figsize=(10, 6))
sns.barplot(data=freq_df, x='frequency', y='word', palette='Blues_r')
plt.title('Top 20 Most Common Words', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

## 4. Model Training & Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Naive Bayes (CNB)': ComplementNB(alpha=0.1),
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    print(f'\n── {name} ──')
    print(classification_report(y_test, y_pred))
    results[name] = model

In [ ]:
# Confusion matrix heatmaps
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, model) in zip(axes, results.items()):
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'],
                ax=ax, linewidths=0.5)
    ax.set_title(f'Confusion Matrix\n{name}', fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## 5. Full Training Pipeline

In [ ]:
from src.model_training import run_training_pipeline

summary = run_training_pipeline(
    data_dir='../data/',
    models_dir='../models/',
    static_dir='../static/'
)

print('\nBest fake news model:', summary['best_fake_model'])
print('Best sentiment model:', summary['best_sentiment_model'])

## 6. Inference Demo

In [ ]:
from src.prediction import get_predictor

predictor = get_predictor(models_dir='../models/', auto_train=False)

test_articles = [
    "Scientists at MIT have published research showing renewable energy adoption grew 18% last year.",
    "SHOCKING: Government secretly admits vaccines contain microchips — whistleblowers reveal all!",
    "The Federal Reserve held interest rates steady following its quarterly policy meeting.",
]

for article in test_articles:
    result = predictor.predict_text(article)
    print(f'Text: {article[:60]}...')
    print(f'  → Fake/Real : {result.fake_label} ({result.fake_confidence*100:.1f}%)')
    print(f'  → Sentiment : {result.sentiment_label} ({result.sentiment_confidence*100:.1f}%)')
    print()

## 7. Word Cloud
Visualize the most common words in the dataset.

In [ ]:
try:
    from wordcloud import WordCloud
    
    text = ' '.join(df_processed['clean_text'].dropna())
    wc = WordCloud(width=900, height=400, background_color='white',
                   colormap='Blues', max_words=150).generate(text)
    
    plt.figure(figsize=(14, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Most Common Words in Dataset', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install wordcloud: pip install wordcloud')